In [43]:
# import libraries
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from scipy.stats import randint

In [44]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from scipy.stats import randint

# Step 1: Load the dataset from URL
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
data = pd.read_csv(url)

# Step 2: Preprocess the data
# Drop unnecessary columns
data = data.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)

# Handle missing values
data['Age'] = data['Age'].fillna(data['Age'].mean())
data['Embarked'] = data['Embarked'].fillna(data['Embarked'].mode()[0])
data['Fare'] = data['Fare'].fillna(data['Fare'].mean())

# Encode categorical variables
data = pd.get_dummies(data, columns=['Sex', 'Embarked'], drop_first=True)

# Step 3: Define features and target
X = data.drop('Survived', axis=1)
y = data['Survived']

# Step 4: Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 5: Hyperparameter tuning with GridSearchCV
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1  # Use all CPU cores
)
grid_search.fit(X_train, y_train)

# Get results
print("GridSearchCV Best Parameters:", grid_search.best_params_)
print("GridSearchCV Best Cross-Validation Score:", grid_search.best_score_)

# Evaluate on test set
best_grid_model = grid_search.best_estimator_
y_pred_grid = best_grid_model.predict(X_test)
print("GridSearchCV Test Accuracy:", accuracy_score(y_test, y_pred_grid))
print("GridSearchCV Classification Report:\n", classification_report(y_test, y_pred_grid))

# Step 6: Hyperparameter tuning with RandomizedSearchCV
param_dist = {
    'n_estimators': randint(100, 500),
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': randint(2, 11),
    'min_samples_leaf': randint(1, 5)
}
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=50,  # Number of random combinations to try
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1
)
random_search.fit(X_train, y_train)

# Get results
print("RandomizedSearchCV Best Parameters:", random_search.best_params_)
print("RandomizedSearchCV Best Cross-Validation Score:", random_search.best_score_)

# Evaluate on test set
best_random_model = random_search.best_estimator_
y_pred_random = best_random_model.predict(X_test)
print("RandomizedSearchCV Test Accuracy:", accuracy_score(y_test, y_pred_random))
print("RandomizedSearchCV Classification Report:\n", classification_report(y_test, y_pred_random))

GridSearchCV Best Parameters: {'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 100}
GridSearchCV Best Cross-Validation Score: 0.832817886339013
GridSearchCV Test Accuracy: 0.8156424581005587
GridSearchCV Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.90      0.85       105
           1       0.83      0.70      0.76        74

    accuracy                           0.82       179
   macro avg       0.82      0.80      0.80       179
weighted avg       0.82      0.82      0.81       179

RandomizedSearchCV Best Parameters: {'max_depth': 30, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 256}
RandomizedSearchCV Best Cross-Validation Score: 0.8300206835418104
RandomizedSearchCV Test Accuracy: 0.8100558659217877
RandomizedSearchCV Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.90      0.85       105
           1     